# Unified Credit Risk Pipeline

## Data Loading

In [1]:
import pandas as pd
import numpy as np
import re
import os
from rapidfuzz import process, utils
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# File Paths
file_3_5 = 'Table No 3.5 Population Group and Bank Group-wise Classification of Outstanding Credit of SCBs According to Occupation.xlsx'
file_restructuring = '13.Loan Subjected to Restructuring and Corporate Debt Restructured.xlsx'
file_npa = '_6.Movement of Non Performing Assets (NPAs) of Scheduled Commercial Banks (1).xlsx'

# Load Raw Data
table_3_5 = pd.read_excel(file_3_5)
table_restructuring = pd.read_excel(file_restructuring)
table_npa = pd.read_excel(file_npa)

print("Data loaded successfully.")

/home/jules/.pyenv/versions/3.12.12/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/jules/.pyenv/versions/3.12.12/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/home/jules/.pyenv/versions/3.12.12/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Data loaded successfully.


## Initial Inspection

In [2]:
def inspect_table(df, name):
    print(f"--- Inspection: {name} ---")
    print(f"Shape: {df.shape}, Columns: {df.columns.tolist()[:5]}...")

inspect_table(table_3_5, "Table 3.5")
inspect_table(table_restructuring, "Restructuring")
inspect_table(table_npa, "NPA Movement")

--- Inspection: Table 3.5 ---
Shape: (356, 21), Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']...
--- Inspection: Restructuring ---
Shape: (1966, 12), Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']...
--- Inspection: NPA Movement ---
Shape: (1988, 10), Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']...


## Cleaning Utilities

In [3]:
def to_camel_case(text):
    if pd.isna(text) or text == "": return "unnamedColumn"
    text = str(text)
    words = re.findall(r'[A-Z]?[a-z0-9]+|[A-Z]+(?=[A-Z][a-z0-9]|\b)', text)
    if not words: words = re.sub(r'[^a-zA-Z0-9]', ' ', text).split()
    if not words: return "unnamedColumn"
    processed = [words[0].lower()]
    for word in words[1:]:
        processed.append(word.capitalize())
    return "".join(processed)

def cleanse_bank_name(val):
    if pd.isna(val): return ""
    s = str(val).upper()
    s = re.sub(r'[^A-Z0-9 ]', '', s)
    return s.strip()

def extract_fiscal_year(val):
    if pd.isna(val): return None
    s = str(val).strip()
    match = re.findall(r'20(\d{2})', s)
    if match: return int("20" + match[-1])
    return None


## First Column Validation & Fix

In [4]:
def validate_first_column(df):
    if df.empty: return df
    first_col = df.columns[0]
    if df[first_col].isna().all():
        df = df.drop(columns=[first_col])
    elif pd.api.types.is_numeric_dtype(df[first_col]):
        df[first_col] = df[first_col].fillna(df[first_col].mean())
    return df

table_3_5 = validate_first_column(table_3_5)
table_restructuring = validate_first_column(table_restructuring)
table_npa = validate_first_column(table_npa)

## Unnamed Column Renaming

In [5]:
def infer_column_names(df):
    new_columns = list(df.columns)
    semantic_map = {
        'occupation': 'occupation', 'accounts': 'noOfAccounts',
        'limit': 'creditLimit', 'outstanding': 'amountOutstanding',
        'restructured': 'restructuredAmount', 'loan': 'loanId'
    }
    for i, col in enumerate(new_columns):
        if "Unnamed" in str(col):
            inferred = None
            for val in df.iloc[:15, i]:
                val_str = str(val).lower()
                for key, mapped in semantic_map.items():
                    if key in val_str:
                        inferred = mapped; break
                if inferred: break
                if len(val_str) > 2 and not val_str.replace('.','').isdigit():
                    inferred = val_str.strip(); break
            if inferred: new_columns[i] = inferred
    df.columns = new_columns
    return df

table_3_5 = infer_column_names(table_3_5)
table_restructuring = infer_column_names(table_restructuring)
table_npa = infer_column_names(table_npa)

## Row-Level Cleaning

In [6]:
def clean_row_levels(df):
    if 1 in df.index:
        row_1_vals = df.loc[1]
        new_cols = list(df.columns)
        for i, val in enumerate(row_1_vals):
            if pd.notna(val) and str(val).strip() != "" and ("Unnamed" in str(new_cols[i]) or "unnamed" in str(new_cols[i]).lower()):
                new_cols[i] = str(val).strip()
        df.columns = new_cols
    if 2 in df.index:
        df = df.drop(index=2)
    return df

table_3_5 = clean_row_levels(table_3_5)
table_restructuring = clean_row_levels(table_restructuring)
table_npa = clean_row_levels(table_npa)

## Column Name Standardization

In [7]:
def standardize_columns(df):
    df.columns = [to_camel_case(col) for col in df.columns]
    new_cols = []
    counts = {}
    for col in df.columns:
        if col in counts:
            counts[col] += 1
            new_cols.append(f"{col}_{counts[col]}")
        else:
            counts[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df

table_3_5 = standardize_columns(table_3_5)
table_restructuring = standardize_columns(table_restructuring)
table_npa = standardize_columns(table_npa)

## Master Consolidation

In [8]:
# Sub-Process 1.1: Temporal Normalization
def apply_temporal(df, name):
    year_col = next((col for col in df.columns if any(x in col.lower() for x in ['year', 'march', 'unnamedColumn'])), df.columns[0])
    df['fiscalYear'] = df[year_col].apply(extract_fiscal_year).ffill()
    df = df[df['fiscalYear'] >= 2018].copy()
    df['fiscalYear'] = df['fiscalYear'].astype(int)
    print(f"Unique fiscalYear values for {name}: {sorted(df['fiscalYear'].unique())}")
    return df

table_3_5 = apply_temporal(table_3_5, "Table 3.5")
table_restructuring = apply_temporal(table_restructuring, "Restructuring")
table_npa = apply_temporal(table_npa, "NPA Movement")

# Sub-Process 1.2: Entity Resolution
# Find bank name column in table_npa (usually index 1 after cleaning)
bank_col_npa = table_npa.columns[1]
table_npa['bankName'] = table_npa[bank_col_npa].map(cleanse_bank_name)
master_bank_list = [b for b in table_npa['bankName'].unique() if len(b) > 3]

def resolve_banks(df, master_list):
    df = df.copy()
    bank_col = next((col for col in df.columns if 'bank' in col.lower() and col != 'bankName'), df.columns[1])
    df['rawBankName'] = df[bank_col].map(cleanse_bank_name)
    unique_names = [n for n in df['rawBankName'].unique() if n]
    mapping = {}
    for name in unique_names:
        if len(name) > 3:
            res = process.extractOne(name, master_list, processor=utils.default_process)
            if res and res[1] > 80:
                mapping[name] = res[0]
            else: mapping[name] = name
        else: mapping[name] = name
    df['bankName'] = df['rawBankName'].map(mapping)
    return df

table_restructuring = resolve_banks(table_restructuring, master_bank_list)

# Sub-Process 1.3: Sectoral Aggregation
def aggregate_3_5(df):
    val_cols = [col for col in df.columns if any(x in col.lower() for x in ['outstanding', 'limit'])]
    # Find occupation col
    occ_col = next(col for col in df.columns if 'occupation' in col.lower())
    id_cols = ['fiscalYear', occ_col]
    melted = pd.melt(df, id_vars=id_cols, value_vars=val_cols, var_name='attr', value_name='val')
    melted['bankGroup'] = melted['attr'].apply(lambda x: 'Public' if 'public' in x.lower() else ('Private' if 'private' in x.lower() else 'Foreign'))
    features = melted.pivot_table(index=['fiscalYear', 'bankGroup'], columns=occ_col, values='val', aggfunc='sum').reset_index()
    features.columns = [to_camel_case(f"credit_{c}") if c not in ['fiscalYear', 'bankGroup'] else c for c in features.columns]
    return features

df_3_5_features = aggregate_3_5(table_3_5)

# Sub-Process 1.4: Incremental Left-Join
def assign_group(name):
    if any(x in str(name).upper() for x in ['STATE BANK', 'CANARA', 'PUNJAB', 'INDIAN', 'BARODA', 'CENTRAL']): return 'Public'
    return 'Private'

table_npa['bankGroup'] = table_npa['bankName'].apply(assign_group)
npa_target_col = table_npa.select_dtypes(include=[np.number]).columns[-1]
table_npa['npaClosingBalance'] = pd.to_numeric(table_npa[npa_target_col], errors='coerce').fillna(0)

rest_val_col = table_restructuring.select_dtypes(include=[np.number]).columns[-1]
table_restructuring['restructuredAmountValue'] = pd.to_numeric(table_restructuring[rest_val_col], errors='coerce').fillna(0)

master_df = pd.merge(table_npa[['fiscalYear', 'bankName', 'bankGroup', 'npaClosingBalance']], 
                     table_restructuring[['fiscalYear', 'bankName', 'restructuredAmountValue']], 
                     on=['fiscalYear', 'bankName'], how='left')

master_df = pd.merge(master_df, df_3_5_features, on=['fiscalYear', 'bankGroup'], how='left')
master_df = master_df.loc[:, ~master_df.columns.duplicated()]

# Sub-Process 1.5: Missing Value Propagation
sector_cols = [c for c in master_df.columns if c.startswith('credit')]
master_df['isImputed'] = 0
for col in sector_cols:
    mask = master_df[col].isnull()
    master_df.loc[mask, 'isImputed'] = 1
    master_df[col] = master_df[col].fillna(master_df.groupby(['fiscalYear', 'bankGroup'])[col].transform('mean'))

num_cols_val = master_df.select_dtypes(include=[np.number]).columns
master_df[num_cols_val] = master_df[num_cols_val].astype(np.float32)
master_df.to_csv('Master_Bank_Data_Consolidated.csv', index=False)


Unique fiscalYear values for Table 3.5: [np.int64(2025)]
Unique fiscalYear values for Restructuring: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Unique fiscalYear values for NPA Movement: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## PHASE 2: FINANCIAL FEATURE ENGINEERING

In [9]:
# Sub-Process 2.1: Mathematical Ratio Calculation
def safe_divide(num, den):
    if den == 0: return 0.0
    return float(num / den)

credit_cols = [col for col in master_df.columns if col.startswith('credit')]
master_df['totalAdvances'] = master_df[credit_cols].sum(axis=1)
master_df['npaRatio'] = master_df.apply(lambda r: safe_divide(r['npaClosingBalance'], r['totalAdvances']), axis=1)

limit_cols = [col for col in master_df.columns if 'limit' in col.lower() and col.startswith('credit')]
out_cols = [col for col in master_df.columns if 'outstanding' in col.lower() and col.startswith('credit')]
master_df['totalLimit'] = master_df[limit_cols].sum(axis=1)
master_df['totalOutstanding'] = master_df[out_cols].sum(axis=1)
master_df['riskWeightRatio'] = master_df.apply(lambda r: safe_divide(r['totalLimit'], r['totalOutstanding']), axis=1)

master_df['totalAssets'] = master_df['totalAdvances'] * 1.2
master_df['restructuringStress'] = master_df.apply(lambda r: safe_divide(r['restructuredAmountValue'], r['totalAssets']), axis=1)

# Sub-Process 2.2: Target Label Synthesis
master_df['isHighRisk'] = (master_df['npaRatio'] > 0.05).astype(int)

# Sub-Process 2.3: Feature Selection & Cleaning
raw_cols = [col for col in master_df.columns if any(x in col.lower() for x in ['balance', 'limit', 'outstanding', 'amount', 'total', 'credit'])]
cols_to_drop = [c for c in raw_cols if c not in ['fiscalYear', 'bankName', 'bankGroup']]
df_final = master_df.drop(columns=cols_to_drop)

ratio_cols = ['npaRatio', 'riskWeightRatio', 'restructuringStress']
for col in ratio_cols:
    df_final[col] = df_final[col].fillna(df_final.groupby('bankGroup')[col].transform('median')).fillna(0)


## PHASE 3: THE PYTORCH DATA PIPELINE

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

df_final = df_final.sort_values(by=['fiscalYear', 'bankName']).reset_index(drop=True)
train_df = df_final[df_final['fiscalYear'] <= 2023].copy()
test_df = df_final[df_final['fiscalYear'] >= 2024].copy()

X_cols = [col for col in df_final.columns if col not in ['fiscalYear', 'bankName', 'bankGroup', 'isHighRisk']]
scaler = StandardScaler()
scaler.fit(train_df[X_cols])

X_train_t = torch.tensor(scaler.transform(train_df[X_cols]), dtype=torch.float32)
X_test_t = torch.tensor(scaler.transform(test_df[X_cols]), dtype=torch.float32)
y_train_t = torch.tensor(train_df['isHighRisk'].values, dtype=torch.float32).reshape(-1, 1)
y_test_t = torch.tensor(test_df['isHighRisk'].values, dtype=torch.float32).reshape(-1, 1)

class BankDefaultDataset(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(BankDefaultDataset(X_train_t, y_train_t), batch_size=16, shuffle=True, num_workers=2, pin_memory=True if torch.cuda.is_available() else False)
test_loader = DataLoader(BankDefaultDataset(X_test_t, y_test_t), batch_size=16, shuffle=False, num_workers=2, pin_memory=True if torch.cuda.is_available() else False)


Using device: cpu


## PHASE 4: ARTIFICIAL NEURAL NETWORK ARCHITECTURE

In [11]:
class CreditRiskANN(nn.Module):
    def __init__(self, input_dim):
        super(CreditRiskANN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=0.3),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(p=0.3),
            nn.Linear(32, 1)
        )
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            if m.out_features == 1: nn.init.xavier_normal_(m.weight)
            else: nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None: nn.init.constant_(m.bias, 0)
    def forward(self, x): return self.model(x)

input_dim = X_train_t.shape[1]
model = CreditRiskANN(input_dim).to(device)


## PHASE 5: MODEL TRAINING

In [12]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

epochs = 50
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        optimizer.zero_grad()
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Average Training Loss: {epoch_loss/len(train_loader):.4f}")


Epoch [10/50], Average Training Loss: 0.5387


Epoch [20/50], Average Training Loss: 0.4161


Epoch [30/50], Average Training Loss: 0.3250


Epoch [40/50], Average Training Loss: 0.2568


Epoch [50/50], Average Training Loss: 0.2053


## PHASE 6: MODEL EVALUATION

In [13]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        logits = model(batch_features)
        preds = (torch.sigmoid(logits) > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_labels.numpy())

print("\n--- Model Evaluation (Test Set) ---")
print(f"Accuracy:  {accuracy_score(all_labels, all_preds):.4f}")
print(f"Precision: {precision_score(all_labels, all_preds, zero_division=0):.4f}")
print(f"Recall:    {recall_score(all_labels, all_preds, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(all_labels, all_preds, zero_division=0):.4f}")



--- Model Evaluation (Test Set) ---
Accuracy:  1.0000
Precision: 0.0000
Recall:    0.0000
F1-Score:  0.0000
